# Day 3 — the validation layer, prototyped

Scratch notebook. Each check is written here first so its output can be reviewed
against the corrupted Bronze, and only then moved into `pipeline/validate.py`.

**Run order matters:** `load_bronze.py` → `corrupt.py` → this notebook. If the
V2.5 cell below shows zeros, Bronze is clean and there is nothing to catch.

Three principles from `DATA_QUALITY_SPEC.md` govern every cell:

1. **Nothing is silently dropped** — every rejected row lands in `quarantine` with a reason.
2. **Identity is never auto-resolved** — suspected duplicates go to a human.
3. **Remediation is recorded, not overwritten** — corrected rows keep the original value.

**DuckDB allows one process on the file at a time.** While this kernel holds `con`, `validate.py` and the other scripts cannot open the database. Run the last cell (`con.close()`) before running any `.py`, or shut the kernel down. Closing the notebook tab does not stop the kernel.

In [1]:
import json
import duckdb
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 60)

DB   = "../data/warehouse/clinical.duckdb"
LOG  = "../data/injected_defects.json"
ASOF = "2026-08-23"          # Decision D7 - fixed as-of date, see cell below
A1C  = "4548-4"

con = duckdb.connect(DB)
def q(sql): return con.sql(sql).df()

injected = json.load(open(LOG))
print("injected defects:", injected["volume"], "| total", injected["total"])

injected defects: {'D1': 40, 'D2': 150, 'D3': 20, 'D4': 8, 'D5': 25, 'D6': 6} | total 249


### Is Bronze actually corrupted? (V2.5, quick)
If any of these is zero, run `corrupt.py` before going further.

In [27]:
q(f"""
SELECT 'D1 dup encounters'     AS defect, count(*) AS n FROM (SELECT Id FROM bronze_encounters GROUP BY PATIENT, Id HAVING count(*) > 1)
UNION ALL SELECT 'D2 orphan observations', count(*) FROM bronze_observations WHERE PATIENT IS NULL
UNION ALL SELECT 'D3 A1c > 20',            count(*) FROM bronze_observations WHERE CODE = '{A1C}' AND TRY_CAST(VALUE AS DOUBLE) > 20
UNION ALL SELECT 'D4 future birth date',   count(*) FROM bronze_patients WHERE BIRTHDATE > '{ASOF}'
UNION ALL SELECT 'D5 discharge < admit',   count(*) FROM bronze_encounters WHERE STOP < START
UNION ALL SELECT 'D6 name+DOB collisions', count(*) FROM (SELECT FIRST, LAST, BIRTHDATE FROM bronze_patients GROUP BY 1,2,3 HAVING count(*) > 1)
""")

,defect,n
0,D1 dup encounters,40
1,D2 orphan observations,150
2,D3 A1c > 20,20
3,D4 future birth date,8
4,D5 discharge < admit,25
5,D6 name+DOB collisions,6


In [ ]:
# - for the encounters, I want to know how the is the patient visit
# bb527977-aea0-113f-540b-93fcbd09ad01


q(sql)

,defect,n
0,D1 dup encounters,40
1,D2 orphan observations,150
2,D3 A1c > 20,0
3,D4 future birth date,0
4,D5 discharge < admit,25
5,D6 name+DOB collisions,6


## Two decisions before any check runs

**D7 — what is "today"?** Fixed at `2026-08-23`, the simulation end date. A run-time
`current_date` would make the README's gap count drift every day someone reads it,
and V1.3 already taught us what unpinned time does. Stored as `ASOF` above and
written into every output table's timestamp column so a run is self-describing.

**D4 — what to do with an A1c of 250.** Explored in the DQ3 section below, with the
numbers for both options, before the rule is written down.

## 3.1 — Build the three output tables first

Schemas from `DATA_QUALITY_SPEC.md`. Built before any check exists, so no check
has an excuse to drop a row quietly. `CREATE OR REPLACE` so the notebook re-runs
cleanly.

In [31]:
con.sql("""
CREATE OR REPLACE TABLE quarantine (
    source_table    VARCHAR,
    source_row_id   VARCHAR,     -- Id, or 'ENCOUNTER|CODE|DATE' for observations
    failure_reason  VARCHAR,
    check_id        VARCHAR,
    quarantined_at  DATE,
    raw_payload     JSON
);
CREATE OR REPLACE TABLE identity_review (
    candidate_a_mrn VARCHAR,
    candidate_b_mrn VARCHAR,
    match_fields    VARCHAR,
    confidence      DOUBLE,
    status          VARCHAR,     -- starts 'pending'; nothing downstream merges
    reviewed_by     VARCHAR,
    reviewed_at     DATE
);
CREATE OR REPLACE TABLE remediation_log (
    source_table     VARCHAR,
    source_row_id    VARCHAR,
    field            VARCHAR,
    original_value   VARCHAR,
    corrected_value  VARCHAR,
    remediation_rule VARCHAR,
    applied_at       DATE
);
""")
q("SELECT table_name, count(*) AS columns FROM information_schema.columns WHERE table_name IN ('quarantine','identity_review','remediation_log') GROUP BY 1")

,table_name,columns
0,identity_review,7
1,quarantine,6
2,remediation_log,7


## 3.2 — DQ1: encounter uniqueness

One row per `(patient_id, encounter_id)`. Duplicates are byte-identical (an interface
replay), so "earliest" means the first one written — lowest `rowid`. Argument for
earliest over latest: it's what the care team actually saw first, and a replayed
message carries no new information by definition.

In [4]:
q("""
SELECT Id, PATIENT, START, count(*) AS copies
FROM bronze_encounters GROUP BY Id, PATIENT, START HAVING count(*) > 1
ORDER BY Id LIMIT 5
""")

,Id,PATIENT,START,copies
0,01e52f9d-3872-9a59-6705-4e8c0d68fd50,01e52f9d-3872-9a59-2200-ed010dfafe82,2007-12-12T10:09:27Z,2
1,09527386-1d9c-f6a2-b4a0-352926e9e802,09527386-1d9c-f6a2-8c55-03eeb927f546,1993-01-11T17:18:06Z,2
2,12c65474-ae7c-3a2e-ca5f-35f798c00024,12c65474-ae7c-3a2e-1b7e-c48f3dae9f3b,2024-02-08T21:08:34Z,2
3,18f1d934-bd81-79d0-7375-fed0e70fcc11,18f1d934-bd81-79d0-83b8-bcbc0f24ecff,2021-05-29T09:41:56Z,2
4,1c0fa1bd-7f20-9e70-8739-99837007656c,1c0fa1bd-7f20-9e70-22c7-f293f24941f5,2022-06-13T23:06:17Z,2


In [39]:
q("""
SELECT *
FROM bronze_encounters
WHERE Id = 'af64b0bf-aad5-9272-929b-10773e4c9dc4'
ORDER BY Id
""")

,Id,START,STOP,PATIENT,ORGANIZATION,PROVIDER,PAYER,ENCOUNTERCLASS,CODE,DESCRIPTION,BASE_ENCOUNTER_COST,TOTAL_CLAIM_COST,PAYER_COVERAGE,REASONCODE,REASONDESCRIPTION,_loaded_at,_source_file
0,af64b0bf-aad5-9272-929b-10773e4c9dc4,2024-03-23T11:00:58Z,2024-03-23T11:54:44Z,af64b0bf-aad5-9272-d456-0b1ceecaa603,9203d0d2-ca7d-3421-9361-974402ffb561,01c19ab0-9680-322d-af81-67b22469de86,b046940f-1664-3047-bca7-dfa76be352a4,wellness,162673000,General examination of patient (procedure),136.80,1586.70,0.00,None,None,2026-09-20 23:47:39.685421-04:00,encounters.csv
1,af64b0bf-aad5-9272-929b-10773e4c9dc4,2024-03-23T11:00:58Z,2024-03-23T11:54:44Z,af64b0bf-aad5-9272-d456-0b1ceecaa603,9203d0d2-ca7d-3421-9361-974402ffb561,01c19ab0-9680-322d-af81-67b22469de86,b046940f-1664-3047-bca7-dfa76be352a4,wellness,162673000,General examination of patient (procedure),136.80,1586.70,0.00,None,None,2026-09-20 23:47:39.685421-04:00,encounters.csv


In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP TABLE rej_dq1 AS
SELECT rowid AS rid, e.*
FROM bronze_encounters e
QUALIFY row_number() OVER (PARTITION BY PATIENT, Id ORDER BY rowid) > 1;

INSERT INTO quarantine
SELECT 'bronze_encounters', Id,b
       'Duplicate encounter row for the same patient and encounter id; earliest copy kept',
       'DQ1', '{ASOF}', to_json(r)
FROM (SELECT * EXCLUDE (rid) FROM rej_dq1) r;
""")
q("SELECT check_id, count(*) AS quarantined FROM quarantine GROUP BY 1")

,check_id,quarantined
0,DQ1,40


## 3.3 — DQ2: referential integrity

Every `observations.PATIENT` must exist in `bronze_patients`. An observation with no
resolvable patient is a lab result nobody will ever see — quarantine it so someone
can go and find out whose it was.

Two flavours: `PATIENT` is null (what D2 injected), or `PATIENT` is set but matches
nobody. Both are orphans; the reason text says which.

In [40]:
con.sql(f"""
CREATE OR REPLACE TEMP TABLE rej_dq2 AS
SELECT o.*
FROM bronze_observations o
LEFT JOIN bronze_patients p ON p.Id = o.PATIENT
WHERE o.PATIENT IS NULL OR p.Id IS NULL;

INSERT INTO quarantine
SELECT 'bronze_observations',
       coalesce(ENCOUNTER, '') || '|' || CODE || '|' || DATE,
       CASE WHEN PATIENT IS NULL THEN 'Observation has no patient identifier'
            ELSE 'Observation patient identifier does not match any patient' END,
       'DQ2', '{ASOF}', to_json(r)
FROM rej_dq2 r;
""")
q("SELECT failure_reason, count(*) FROM quarantine WHERE check_id = 'DQ2' GROUP BY 1")

,failure_reason,count_star()
0,Observation has no patient identifier,150


In [42]:
q("""
    select * from quarantine where check_id = 'DQ2' limit 2

""")

,source_table,source_row_id,failure_reason,check_id,quarantined_at,raw_payload
0,bronze_observations,a677755c-46ef-ae71-fd9b-8d509bd67b12|8480-6|2019-05-24T0...,Observation has no patient identifier,DQ2,2026-08-23,"{""DATE"":""2019-05-24T04:42:46Z"",""PATIENT"":null,""ENCOUNTER..."
1,bronze_observations,1e528593-92f0-e5b7-8177-d5e61cd6c4ff|8867-4|2019-10-30T1...,Observation has no patient identifier,DQ2,2026-08-23,"{""DATE"":""2019-10-30T13:17:08Z"",""PATIENT"":null,""ENCOUNTER..."


## 3.4 — DQ3: A1c plausibility, and Decision D4

Day 1 found that the spec's floor of 3.0 flags 951 clean values. **Range is revised
to 2.0–20.0.** First, look at what is actually out of range now:

In [44]:
q(f"""
SELECT CASE WHEN v IS NULL THEN 'non-numeric'
            WHEN v < 2.0  THEN 'below 2.0'
            WHEN v > 20.0 THEN 'above 20.0' ELSE 'in range' END AS bucket,
       count(*) AS n, min(v) AS lo, max(v) AS hi
FROM (SELECT TRY_CAST(VALUE AS DOUBLE) AS v FROM bronze_observations WHERE CODE = '{A1C}')
GROUP BY 1 ORDER BY n DESC
""")

,bucket,n,lo,hi
0,in range,8921,2.3,8.8
1,above 20.0,20,250.0,250.0


### What is a 250, really?

A1c is a percentage; nothing biological produces 250. Blood glucose in mg/dL,
however, lands there all the time. The ADA publishes a mapping between A1c and
*estimated average glucose*: `eAG (mg/dL) = 28.7 × A1c − 46.7`. Inverted:
`A1c = (eAG + 46.7) / 28.7`, so 250 mg/dL ↔ **10.3 %** — a plausible, poorly
controlled diabetic.

**Option A — remediate.** Treat any A1c in the glucose range (40–600) as a
mis-keyed glucose, convert with the eAG formula, keep the original in
`remediation_log`, mark the Silver row `remediated`. Exercises the third principle
and gives the app a remediation to show.

**Option B — quarantine, don't guess.** A single glucose reading is not an A1c;
converting it invents a lab result. Route to a human instead.

The difference is visible in the gap count because D3 was sampled from cohort
patients. Both computed here so the choice is made on numbers:

In [45]:
q(f"""
WITH a1c AS (
    SELECT PATIENT, DATE, TRY_CAST(VALUE AS DOUBLE) AS v
    FROM bronze_observations WHERE CODE = '{A1C}' AND PATIENT IS NOT NULL
), cohort AS (
    SELECT DISTINCT PATIENT FROM bronze_conditions
    WHERE CODE IN ('44054006','127013003','90781000119102','157141000119108',
                   '368581000119106','1551000119108','97331000119101','1501000119109')
), last_valid AS (          -- option B: the 250s are gone
    SELECT c.PATIENT, max(a.DATE) AS last_a1c FROM cohort c
    LEFT JOIN a1c a ON a.PATIENT = c.PATIENT AND a.v BETWEEN 2.0 AND 20.0 GROUP BY 1
), last_any AS (            -- option A: the 250s count as tests (with a corrected value)
    SELECT c.PATIENT, max(a.DATE) AS last_a1c FROM cohort c
    LEFT JOIN a1c a ON a.PATIENT = c.PATIENT AND (a.v BETWEEN 2.0 AND 20.0 OR a.v BETWEEN 40 AND 600) GROUP BY 1
)
SELECT 'A  remediate (250 -> 10.3%)' AS option,
       count(*) FILTER (WHERE last_a1c IS NULL OR last_a1c < '2025-08-23') AS open_gaps FROM last_any
UNION ALL
SELECT 'B  quarantine (250 removed)',
       count(*) FILTER (WHERE last_a1c IS NULL OR last_a1c < '2025-08-23') FROM last_valid
""")

,option,open_gaps
0,A remediate (250 -> 10.3%),69
1,B quarantine (250 removed),69


### D4 as implemented below: **Option A, remediate — with the rule written down**

(The cell above shows both options give the same gap count on this extract — none of
the 20 corrupted A1cs was a patient's most recent result inside the 12-month window.
So D4 is decided on principle here, not on numbers. If the demo needs the decision
to *visibly* move the count, `corrupt.py` could sample D3 from each cohort patient's
latest A1c — defensible, since the latest result is where a unit error does the most
damage — but that is a change to the injection, and it should be made knowingly.)

- `value > 20 AND 40 ≤ value ≤ 600` → treated as mg/dL glucose, converted with the
  eAG formula, rounded to 1 dp. Rule name `A1C_MGDL_TO_PCT_EAG`. Original kept.
- `value < 2.0`, `value > 600`, or non-numeric → quarantined. Not guessed.

Why A over B: the remediation is *reversible and visible* — `remediation_log`
holds the original, the Silver row is flagged, and a reviewer who disagrees can
exclude `remediated` rows in one `WHERE` clause. Option B throws the row away
for everyone. In an interview, the honest framing is: "I applied a published
conversion, logged it, and made it trivially reversible — and I'd want a
clinician to sign off on the rule before it ran on real data."

If you'd rather ship B, it is one line: move the 40–600 band from `rem_dq3` into
`rej_dq3`.

In [46]:
con.sql(f"""
CREATE OR REPLACE TEMP TABLE a1c AS
SELECT o.*, TRY_CAST(VALUE AS DOUBLE) AS v,
       coalesce(ENCOUNTER, '') || '|' || CODE || '|' || DATE AS row_key
FROM bronze_observations o WHERE CODE = '{A1C}';

-- remediate: plausible glucose keyed into a percent field
CREATE OR REPLACE TEMP TABLE rem_dq3 AS
SELECT *, round((v + 46.7) / 28.7, 1) AS corrected
FROM a1c WHERE v > 20.0 AND v BETWEEN 40 AND 600;

-- reject: everything else out of range. Raw Bronze columns only, so raw_payload stays faithful.
CREATE OR REPLACE TEMP TABLE rej_dq3 AS
SELECT * EXCLUDE (v, row_key) FROM a1c
WHERE v IS NULL OR v < 2.0 OR (v > 20.0 AND v NOT BETWEEN 40 AND 600);

INSERT INTO remediation_log
SELECT 'bronze_observations', row_key, 'VALUE', VALUE, corrected::VARCHAR,
       'A1C_MGDL_TO_PCT_EAG: value in glucose range keyed into a percent field; A1c = (value + 46.7) / 28.7',
       '{ASOF}'
FROM rem_dq3;

INSERT INTO quarantine
SELECT 'bronze_observations', coalesce(ENCOUNTER, '') || '|' || CODE || '|' || DATE,
       CASE WHEN TRY_CAST(VALUE AS DOUBLE) IS NULL THEN 'A1c value is not numeric'
            WHEN TRY_CAST(VALUE AS DOUBLE) < 2.0   THEN 'A1c below plausible floor of 2.0 %'
            ELSE 'A1c above 20 % and not in a glucose range; cannot infer intended value' END,
       'DQ3', '{ASOF}', to_json(r)
FROM rej_dq3 r;
""")
print("remediated:", con.sql("SELECT count(*) FROM rem_dq3").fetchone()[0], "  quarantined by DQ3:", con.sql("SELECT count(*) FROM rej_dq3").fetchone()[0])
q("SELECT source_row_id, original_value, corrected_value, remediation_rule[1:22] AS rule FROM remediation_log LIMIT 5")

remediated: 20   quarantined by DQ3: 0


,source_row_id,original_value,corrected_value,rule
0,f870f04f-d88f-1675-6e57-e51ff4f603b0|4548-4|2023-07-27T0...,250,10.3,A1C_MGDL_TO_PCT_EAG: v
1,94397896-83cf-8014-d293-36ef8f8ca9c6|4548-4|2026-03-22T2...,250,10.3,A1C_MGDL_TO_PCT_EAG: v
2,95feb885-62f0-2deb-2452-e500473d99a4|4548-4|2021-09-26T1...,250,10.3,A1C_MGDL_TO_PCT_EAG: v
3,df02c43f-cb47-ef47-fc05-6abb282f0410|4548-4|1991-12-09T1...,250,10.3,A1C_MGDL_TO_PCT_EAG: v
4,689b8201-5022-1f9e-b22e-3a55efbdee51|4548-4|2017-10-26T0...,250,10.3,A1C_MGDL_TO_PCT_EAG: v


## 3.5 — DQ4: birth date sanity

In the past (relative to `ASOF`, not the wall clock — that's D7) and implied age ≤ 120.

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP TABLE rej_dq4 AS
SELECT * FROM bronze_patients
WHERE TRY_CAST(BIRTHDATE AS DATE) IS NULL
   OR BIRTHDATE::DATE > DATE '{ASOF}'
   OR date_diff('year', BIRTHDATE::DATE, DATE '{ASOF}') > 120;

INSERT INTO quarantine
SELECT 'bronze_patients', Id,
       CASE WHEN TRY_CAST(BIRTHDATE AS DATE) IS NULL THEN 'Birth date is not a valid date'
            WHEN BIRTHDATE::DATE > DATE '{ASOF}'    THEN 'Birth date is after the as-of date'
            ELSE 'Implied age exceeds 120 years' END,
       'DQ4', '{ASOF}', to_json(r)
FROM rej_dq4 r;
""")
q("SELECT failure_reason, count(*) FROM quarantine WHERE check_id = 'DQ4' GROUP BY 1")

,failure_reason,count_star()
0,Birth date is after the as-of date,8


In [48]:
q("""select * from quarantine WHERE check_id = 'DQ4'""")

,source_table,source_row_id,failure_reason,check_id,quarantined_at,raw_payload
0,bronze_patients,425a2728-cebb-9101-7b3b-f1f9f8c87b48,Birth date is after the as-of date,DQ4,2026-08-23,"{""Id"":""425a2728-cebb-9101-7b3b-f1f9f8c87b48"",""BIRTHDATE""..."
1,bronze_patients,662496ff-c3fd-e92b-01ab-91bc2e079ddf,Birth date is after the as-of date,DQ4,2026-08-23,"{""Id"":""662496ff-c3fd-e92b-01ab-91bc2e079ddf"",""BIRTHDATE""..."
2,bronze_patients,3c72e166-bb7a-9132-32e5-3582e34b2174,Birth date is after the as-of date,DQ4,2026-08-23,"{""Id"":""3c72e166-bb7a-9132-32e5-3582e34b2174"",""BIRTHDATE""..."
3,bronze_patients,dae1f111-05b1-8d1d-969f-f48ec0396e8f,Birth date is after the as-of date,DQ4,2026-08-23,"{""Id"":""dae1f111-05b1-8d1d-969f-f48ec0396e8f"",""BIRTHDATE""..."
4,bronze_patients,cf5ac1b8-e980-f0de-fb02-6b0bbbbf172c,Birth date is after the as-of date,DQ4,2026-08-23,"{""Id"":""cf5ac1b8-e980-f0de-fb02-6b0bbbbf172c"",""BIRTHDATE""..."
5,bronze_patients,ba9056d8-c2d7-6f8f-a320-09cb66f9eabf,Birth date is after the as-of date,DQ4,2026-08-23,"{""Id"":""ba9056d8-c2d7-6f8f-a320-09cb66f9eabf"",""BIRTHDATE""..."
6,bronze_patients,d11d8161-498d-63f9-e17d-d403a0cd5875,Birth date is after the as-of date,DQ4,2026-08-23,"{""Id"":""d11d8161-498d-63f9-e17d-d403a0cd5875"",""BIRTHDATE""..."
7,bronze_patients,81bb2cc1-8556-ccad-8738-00d187ffb540,Birth date is after the as-of date,DQ4,2026-08-23,"{""Id"":""81bb2cc1-8556-ccad-8738-00d187ffb540"",""BIRTHDATE""..."


In [50]:
# q("""select * from bronze_patients where id = '425a2728-cebb-9101-7b3b-f1f9f8c87b48' """)

## 3.6 — DQ5: encounter chronology

Discharge ≥ admission. **The null trap:** an open encounter has no `STOP`, and
`NULL < START` is *unknown*, not true — so `WHERE STOP < START` leaves it alone,
which is what we want. But `WHERE NOT (STOP >= START)` would *also* leave it
alone for the same reason, so the two forms are only accidentally equivalent.
The condition is written so that null discharge is explicitly kept.

There are 0 open encounters in this extract (checked Day 2), but the guard is
written for the day there aren't.

In [51]:
con.sql(f"""
CREATE OR REPLACE TEMP TABLE rej_dq5 AS
SELECT * FROM bronze_encounters
WHERE STOP IS NOT NULL AND STOP <> ''           -- open encounters are valid, keep them
  AND TRY_CAST(STOP AS TIMESTAMP) < TRY_CAST(START AS TIMESTAMP)
  AND rowid NOT IN (SELECT rid FROM rej_dq1);   -- a row can only be rejected once

INSERT INTO quarantine
SELECT 'bronze_encounters', Id,
       'Discharge timestamp is before admission timestamp',
       'DQ5', '{ASOF}', to_json(r)
FROM rej_dq5 r;
""")
q("SELECT check_id, count(*) FROM quarantine WHERE check_id = 'DQ5' GROUP BY 1")

,check_id,count_star()
0,DQ5,25


In [59]:
#q("""select * FROM quarantine WHERE check_id = 'DQ5' """)
q("""
SELECT *, 
       DATE_DIFF(
           'minute', 
           TRY_CAST(START AS TIMESTAMP), 
           TRY_CAST(STOP AS TIMESTAMP)
       ) AS duration_minutes
FROM bronze_encounters 
WHERE id = '6d907ed2-a003-7509-d65c-1888079f862e'
""")




,Id,START,STOP,PATIENT,ORGANIZATION,PROVIDER,PAYER,ENCOUNTERCLASS,CODE,DESCRIPTION,BASE_ENCOUNTER_COST,TOTAL_CLAIM_COST,PAYER_COVERAGE,REASONCODE,REASONDESCRIPTION,_loaded_at,_source_file,duration_minutes
0,6d907ed2-a003-7509-d65c-1888079f862e,2022-12-04T15:39:59Z,2022-12-04T12:35:59Z,6d907ed2-a003-7509-190d-325cf4c79b5f,05fcf001-ab24-3d7d-85f7-f64942d82738,3bb79b69-b91c-3c21-b451-5c7903fe1b0c,a735bf55-83e9-331a-899d-a82a60b9f60c,ambulatory,185347001,Encounter for problem (procedure),85.55,1745.73,1396.58,431857002,Chronic kidney disease stage 4 (disorder),2026-09-20 23:47:39.685421-04:00,encounters.csv,-184


## 3.7 — DQ6: identity review

Two patients with the same first name, last name and birth date. They go to
`identity_review` as `pending` with a confidence score. **Both stay in Silver as
separate people.** A wrong merge combines two medication lists — a patient safety
event — so the design routes to a human and that is the correct design, not a
limitation.

Confidence: 0.70 for name + DOB, +0.25 if SSN also matches, +0.05 if address does.

In [60]:
con.sql(f"""
INSERT INTO identity_review
SELECT a.Id, b.Id,
       'first_name,last_name,birth_date'
         || CASE WHEN a.SSN = b.SSN THEN ',ssn' ELSE '' END
         || CASE WHEN a.ADDRESS = b.ADDRESS THEN ',address' ELSE '' END,
       0.70 + CASE WHEN a.SSN = b.SSN THEN 0.25 ELSE 0 END
            + CASE WHEN a.ADDRESS = b.ADDRESS THEN 0.05 ELSE 0 END,
       'pending', NULL, NULL
FROM bronze_patients a
JOIN bronze_patients b
  ON a.FIRST = b.FIRST AND a.LAST = b.LAST AND a.BIRTHDATE = b.BIRTHDATE AND a.Id < b.Id;
""")
q("SELECT candidate_a_mrn[1:8] AS a, candidate_b_mrn[1:8] AS b, match_fields, confidence, status FROM identity_review")

,a,b,match_fields,confidence,status
0,93a5bbce,ab5a9b62,"first_name,last_name,birth_date,ssn,address",1.0,pending
1,9c5895b3,aa081e8d,"first_name,last_name,birth_date,ssn,address",1.0,pending
2,4917c584,b5c36d2a,"first_name,last_name,birth_date,ssn,address",1.0,pending
3,bb610fec,d119c336,"first_name,last_name,birth_date,ssn,address",1.0,pending
4,1782a49d,ee2fdada,"first_name,last_name,birth_date,ssn,address",1.0,pending
5,3c9cff8e,5c92cf2b,"first_name,last_name,birth_date,ssn,address",1.0,pending


## 3.8 — Build Silver

Surviving rows only, with real types. `_dq_status` is `clean` or `remediated`.
Column names follow `DATA_DICTIONARY.md`; a few extra source columns are kept
where the app will need them (names, descriptions, encounter link).

`observations.value` is `DOUBLE` — but 315,450 observations carry text results
(smoking status, survey answers). `TRY_CAST` would null those silently, which is
V3.10's disguised drop. So `value_text` keeps the original for every row and
`value` is only populated where the source was numeric.

In [61]:
con.sql(f"""
CREATE OR REPLACE TABLE silver_patients AS
SELECT Id AS patient_id, Id AS mrn,               -- Synthea has no MRN; Id plays the role
       BIRTHDATE::DATE AS birth_date, TRY_CAST(DEATHDATE AS DATE) AS death_date,
       GENDER AS sex, FIRST AS first_name, LAST AS last_name,
       'clean' AS _dq_status
FROM bronze_patients WHERE Id NOT IN (SELECT Id FROM rej_dq4);

CREATE OR REPLACE TABLE silver_encounters AS
SELECT Id AS encounter_id, PATIENT AS patient_id,
       START::TIMESTAMP AS admission_ts, TRY_CAST(STOP AS TIMESTAMP) AS discharge_ts,
       ENCOUNTERCLASS AS encounter_type, CODE AS snomed_code, DESCRIPTION AS description,
       'clean' AS _dq_status
FROM bronze_encounters
WHERE rowid NOT IN (SELECT rid FROM rej_dq1) AND rowid NOT IN (SELECT rowid FROM rej_dq5);

CREATE OR REPLACE TABLE silver_conditions AS
SELECT PATIENT AS patient_id, ENCOUNTER AS encounter_id, CODE AS snomed_code, DESCRIPTION AS description,
       START::DATE AS onset_date, TRY_CAST(STOP AS DATE) AS resolved_date,
       'clean' AS _dq_status
FROM bronze_conditions;

CREATE OR REPLACE TABLE silver_observations AS
SELECT o.PATIENT AS patient_id, o.ENCOUNTER AS encounter_id, o.CODE AS loinc_code, o.DESCRIPTION AS description,
       coalesce(m.corrected, TRY_CAST(o.VALUE AS DOUBLE)) AS value,
       o.VALUE AS value_text, o.UNITS AS unit, o.DATE::TIMESTAMP AS observed_at,
       CASE WHEN m.row_key IS NOT NULL THEN 'remediated' ELSE 'clean' END AS _dq_status
FROM bronze_observations o
LEFT JOIN rem_dq3 m ON m.row_key = coalesce(o.ENCOUNTER, '') || '|' || o.CODE || '|' || o.DATE
WHERE o.rowid NOT IN (SELECT rowid FROM rej_dq2) AND o.rowid NOT IN (SELECT rowid FROM rej_dq3);

CREATE OR REPLACE TABLE silver_medications AS
SELECT PATIENT AS patient_id, ENCOUNTER AS encounter_id, CODE AS rxnorm_code, DESCRIPTION AS description,
       START::DATE AS start_date, TRY_CAST(STOP AS DATE) AS end_date,
       'clean' AS _dq_status
FROM bronze_medications;
""")
q("SELECT table_name, column_name, data_type FROM information_schema.columns WHERE table_name = 'silver_observations'")

,table_name,column_name,data_type
0,silver_observations,patient_id,VARCHAR
1,silver_observations,encounter_id,VARCHAR
2,silver_observations,loinc_code,VARCHAR
3,silver_observations,description,VARCHAR
4,silver_observations,value,DOUBLE
5,silver_observations,value_text,VARCHAR
6,silver_observations,unit,VARCHAR
7,silver_observations,observed_at,TIMESTAMP
8,silver_observations,_dq_status,VARCHAR


In [62]:
q("""select * from silver_observations limit 10""")

,patient_id,encounter_id,loinc_code,description,value,value_text,unit,observed_at,_dq_status
0,a677755c-46ef-ae71-bf00-afedbc06333a,a677755c-46ef-ae71-fd9b-8d509bd67b12,72514-3,Pain severity - 0-10 verbal numeric rating [Score] - Rep...,1.0,1.0,{score},2019-05-24 04:42:46,clean
1,a677755c-46ef-ae71-bf00-afedbc06333a,a677755c-46ef-ae71-fd9b-8d509bd67b12,29463-7,Body Weight,82.4,82.4,kg,2019-05-24 04:42:46,clean
2,a677755c-46ef-ae71-bf00-afedbc06333a,a677755c-46ef-ae71-fd9b-8d509bd67b12,39156-5,Body mass index (BMI) [Ratio],27.9,27.9,kg/m2,2019-05-24 04:42:46,clean
3,a677755c-46ef-ae71-bf00-afedbc06333a,a677755c-46ef-ae71-fd9b-8d509bd67b12,8462-4,Diastolic Blood Pressure,71.0,71.0,mm[Hg],2019-05-24 04:42:46,clean
4,NaN,a677755c-46ef-ae71-fd9b-8d509bd67b12,8480-6,Systolic Blood Pressure,107.0,107.0,mm[Hg],2019-05-24 04:42:46,clean
5,a677755c-46ef-ae71-bf00-afedbc06333a,a677755c-46ef-ae71-fd9b-8d509bd67b12,8867-4,Heart rate,73.0,73.0,/min,2019-05-24 04:42:46,clean
6,a677755c-46ef-ae71-bf00-afedbc06333a,a677755c-46ef-ae71-fd9b-8d509bd67b12,9279-1,Respiratory rate,13.0,13.0,/min,2019-05-24 04:42:46,clean
7,a677755c-46ef-ae71-bf00-afedbc06333a,a677755c-46ef-ae71-fd9b-8d509bd67b12,2339-0,Glucose [Mass/volume] in Blood,88.8,88.8,mg/dL,2019-05-24 04:42:46,clean
8,a677755c-46ef-ae71-bf00-afedbc06333a,a677755c-46ef-ae71-fd9b-8d509bd67b12,6299-2,Urea nitrogen [Mass/volume] in Blood,7.4,7.4,mg/dL,2019-05-24 04:42:46,clean
9,a677755c-46ef-ae71-bf00-afedbc06333a,a677755c-46ef-ae71-fd9b-8d509bd67b12,38483-4,Creatinine [Mass/volume] in Blood,0.7,0.7,mg/dL,2019-05-24 04:42:46,clean


## 3.10 — Assert no silent drops (V3.1)

For every table: `bronze = silver + quarantine`. Remediated rows are in Silver
(and in `remediation_log`), not in quarantine, so they count once on the Silver
side. This is the cell that turns principle 1 from a sentence into a test.

In [63]:
TABLES = ["patients", "encounters", "conditions", "observations", "medications"]
recon = q(" UNION ALL ".join(f"""
    SELECT '{t}' AS name,
           (SELECT count(*) FROM bronze_{t}) AS bronze,
           (SELECT count(*) FROM silver_{t}) AS silver,
           (SELECT count(*) FROM quarantine WHERE source_table = 'bronze_{t}') AS quarantined
""" for t in TABLES))
recon["balances"] = recon.bronze == recon.silver + recon.quarantined
assert recon.balances.all(), "A check dropped rows silently"
recon

,name,bronze,silver,quarantined,balances
0,patients,1159,1151,8,True
1,encounters,67795,67730,65,True
2,conditions,40811,40811,0,True
3,observations,870510,870360,150,True
4,medications,59273,59273,0,True


## 3.9 — Catch rate, measured against ground truth (V3.4, V3.5)

For each entry in `injected_defects.json`: did it land in the right destination
table **with the right check id**? Caught-by-the-wrong-check is a coincidence,
not a working check, so the match requires both.

In [64]:
entries = pd.DataFrame([
    {"defect": e["defect"], "table": e["table"],
     "key": "|".join(str(v) for v in e["key"].values()) if e["defect"] in ("D2","D3") else e["key"]["Id"]}
    for e in injected["entries"]
])
con.register("entries", entries)
catch = q("""
WITH found AS (
    SELECT source_row_id AS key, check_id AS caught_by FROM quarantine
    UNION ALL SELECT source_row_id, 'DQ3' FROM remediation_log
    UNION ALL SELECT candidate_a_mrn, 'DQ6' FROM identity_review   -- the derived Id can sort either side of its source
    UNION ALL SELECT candidate_b_mrn, 'DQ6' FROM identity_review
)
SELECT e.defect, 'DQ' || e.defect[2] AS expected_check,
       count(*) AS injected,
       count(f.key) FILTER (WHERE f.caught_by = 'DQ' || e.defect[2]) AS caught_by_right_check,
       count(f.key) FILTER (WHERE f.caught_by <> 'DQ' || e.defect[2]) AS caught_by_other
FROM entries e LEFT JOIN found f ON f.key = e.key
GROUP BY 1, 2 ORDER BY 1
""")
catch["type_caught"] = catch.caught_by_right_check > 0
print(f"catch rate by type: {catch.type_caught.sum()} of 6   |   by row: {catch.caught_by_right_check.sum()} of {catch.injected.sum()}")
catch

catch rate by type: 6 of 6   |   by row: 249 of 249


,defect,expected_check,injected,caught_by_right_check,caught_by_other,type_caught
0,D1,DQ1,40,40,0,True
1,D2,DQ2,150,150,0,True
2,D3,DQ3,20,20,0,True
3,D4,DQ4,8,8,0,True
4,D5,DQ5,25,25,0,True
5,D6,DQ6,6,6,0,True


## The rest of the V3.x checks

In [65]:
print("V3.2 every quarantine row labelled:")
display(q("SELECT check_id, failure_reason, count(*) AS n FROM quarantine GROUP BY 1,2 ORDER BY 1"))
print("V3.6 open encounters untouched (null discharge in Silver):", con.sql("SELECT count(*) FROM silver_encounters WHERE discharge_ts IS NULL").fetchone()[0], "- none existed in Bronze either")
print("V3.7 both D6 patients still in Silver:", con.sql("SELECT count(*) FROM silver_patients WHERE patient_id IN (SELECT candidate_a_mrn FROM identity_review UNION ALL SELECT candidate_b_mrn FROM identity_review)").fetchone()[0], "of", 2 * con.sql("SELECT count(*) FROM identity_review").fetchone()[0], "| status:", q("SELECT DISTINCT status FROM identity_review").status.tolist())
print("V3.8 remediation preserved original:", con.sql("SELECT count(*) FROM remediation_log WHERE original_value IS NULL OR corrected_value IS NULL").fetchone()[0], "rows missing a value |  Silver rows flagged remediated:", con.sql("SELECT count(*) FROM silver_observations WHERE _dq_status='remediated'").fetchone()[0])
print("V3.11 no Silver table larger than its Bronze:", (recon.silver <= recon.bronze).all())

V3.2 every quarantine row labelled:


,check_id,failure_reason,n
0,DQ1,Duplicate encounter row for the same patient and encount...,40
1,DQ2,Observation has no patient identifier,150
2,DQ4,Birth date is after the as-of date,8
3,DQ5,Discharge timestamp is before admission timestamp,25


V3.6 open encounters untouched (null discharge in Silver): 0 - none existed in Bronze either
V3.7 both D6 patients still in Silver: 12 of 12 | status: ['pending']
V3.8 remediation preserved original: 0 rows missing a value |  Silver rows flagged remediated: 20
V3.11 no Silver table larger than its Bronze: True


In [66]:
print("V3.10 - nulls created by casting, per column (bronze -> silver). Differences must be explained by a quarantine or by value_text.")
q("""
SELECT 'observations.value'    AS col, (SELECT count(*) FROM bronze_observations WHERE TRY_CAST(VALUE AS DOUBLE) IS NULL) AS bronze_nulls, (SELECT count(*) FROM silver_observations WHERE value IS NULL) AS silver_nulls, 'text results; original kept in value_text' AS explanation
UNION ALL SELECT 'encounters.discharge_ts', (SELECT count(*) FROM bronze_encounters WHERE STOP IS NULL), (SELECT count(*) FROM silver_encounters WHERE discharge_ts IS NULL), 'no open encounters in this extract'
UNION ALL SELECT 'conditions.resolved_date', (SELECT count(*) FROM bronze_conditions WHERE STOP IS NULL), (SELECT count(*) FROM silver_conditions WHERE resolved_date IS NULL), 'null = still active, by design'
UNION ALL SELECT 'patients.death_date',     (SELECT count(*) FROM bronze_patients WHERE DEATHDATE IS NULL), (SELECT count(*) FROM silver_patients WHERE death_date IS NULL), 'null = alive; differs by the 8 DQ4 rejects'
""")

V3.10 - nulls created by casting, per column (bronze -> silver). Differences must be explained by a quarantine or by value_text.


,col,bronze_nulls,silver_nulls,explanation
0,observations.value,315450,315439,text results; original kept in value_text
1,encounters.discharge_ts,0,0,no open encounters in this extract
2,conditions.resolved_date,10532,10532,"null = still active, by design"
3,patients.death_date,1005,997,null = alive; differs by the 8 DQ4 rejects


In [67]:
print("V3.3 - five random quarantine rows traced back to Bronze:")
q("""
SELECT source_table, check_id, source_row_id[1:24] AS row_id, (raw_payload::VARCHAR)[1:70] AS payload
FROM quarantine USING SAMPLE 5 ROWS (reservoir, 1)
""")

V3.3 - five random quarantine rows traced back to Bronze:


,source_table,check_id,row_id,payload
0,bronze_observations,DQ2,36c9879a-06f4-d01b-7447-,"{""DATE"":""2018-11-19T23:42:11Z"",""PATIENT"":null,""ENCOUNTER..."
1,bronze_observations,DQ2,cf9f648b-3742-5105-e7b9-,"{""DATE"":""2020-06-18T14:27:20Z"",""PATIENT"":null,""ENCOUNTER..."
2,bronze_observations,DQ2,e29600a7-7eec-1b43-2402-,"{""DATE"":""1980-03-04T14:40:22Z"",""PATIENT"":null,""ENCOUNTER..."
3,bronze_observations,DQ2,5599ae5e-abfd-73d6-15ec-,"{""DATE"":""2024-08-23T16:24:20Z"",""PATIENT"":null,""ENCOUNTER..."
4,bronze_encounters,DQ5,89f8ab28-e0ca-7c4d-bf44-,"{""Id"":""89f8ab28-e0ca-7c4d-bf44-9b440cb486d5"",""START"":""20..."


## Ready to move to `validate.py`

What settled here and should port as-is: the three table schemas, one `rej_*` /
`rem_*` temp table per check, Silver built as "Bronze minus rejects", the
reconciliation assert, and the ground-truth join for catch rate.

What was decided along the way and needs recording: **D4** (remediate via eAG,
rule `A1C_MGDL_TO_PCT_EAG`), **D7** (`ASOF = 2026-08-23`), and the DQ3 range
revision to 2.0–20.0 in `DATA_QUALITY_SPEC.md`.

## Release the database

Always the last cell. Frees the file lock so `validate.py` can run.

In [ ]:
con.close()
print("connection closed - the .py scripts can run now")